# M3 · RAG con tool use — triaje de acciones constitucionales

M2 dejó debilidades medidas, no supuestas, y son exactamente lo que un RAG
puede cerrar: **recuperar el texto de la norma en vez de recordarlo**, y
**recuperar la causal de improcedencia en vez de deducirla**.

Se comparan cuatro sistemas con el mismo harness de M2:

| | sistema | qué aporta |
|---|---|---|
| 1 | fine-tuned solo | baseline de M1, ya medido |
| 2 | fine-tuned + RAG ingenuo | punto de partida del delta de retrieval |
| 3 | fine-tuned + RAG avanzado | la estrategia que gane la regla de decisión |
| 4 | agente con tool use + motor de reglas | el techo: enrutamiento determinista |

**La sección de recuperación corre en CPU.** Solo los brazos 2 y 3 necesitan
GPU, porque cargan el modelo ajustado. Sin GPU el notebook mide recuperación,
herramientas y el brazo 4, y lo dice en vez de fallar.

In [ ]:
# Instalacion. Los pins son los de `requirements.txt` (las versiones de las
# corridas que produjeron las cifras del reporte).
#
# RAGAS SE INSTALA AQUI, EN LA PRIMERA CELDA, Y NO DONDE SE USA: instalarlo a
# mitad actualizaria paquetes que `torch`, `datasets` y `sentence_transformers`
# ya importaron, y el fallo saldria dentro de ragas, lejos de su causa.
# EL PIN QUE IMPORTA ES `langchain-community==0.3.27`: `ragas/llms/base.py`
# importa `langchain_community.chat_models.vertexai`, que la 0.4.x elimino.
!pip install -q groq==1.7.0 sentence-transformers==5.7.0 rank_bm25==0.2.2 ragas==0.2.15 "langchain-community==0.3.27" langchain-openai 2>/dev/null

# Colab trae torchao 0.10.0 y peft >=0.21 exige >0.16: al cargar el adaptador
# LoRA lanza ImportError DESPUES de bajar los 15 GB del modelo base. Aqui no se
# usa torchao (el brazo 3 corre en bf16), asi que se desinstala antes.
!pip uninstall -y -q torchao

import importlib.util
assert importlib.util.find_spec("torchao") is None, (
    "torchao sigue instalado: Entorno -> Reiniciar sesion y corre esta celda de nuevo")

import torch
HAY_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAY_GPU else "no hay — se omiten los brazos 2 y 3")

## 0 · Proyecto, clave y corrida

In [ ]:
# Monta Drive y pone la raiz del proyecto en sys.path: hasta aqui `amparo`
# no se puede importar. Todo lo demas del arranque lo hace `m3.preparar()`.
import sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # fuera de Colab: el repo es el directorio actual o su padre
CANDIDATAS = [Path("/content/drive/MyDrive/agenteAmparo"),
              Path("/content/drive/MyDrive/Colab Notebooks/agenteAmparo"),
              Path("/content/agenteAmparo"), Path("."), Path("..")]
RAIZ = next((p.resolve() for p in CANDIDATAS if (p / "amparo" / "nucleo" / "schema.py").exists()), None)
if RAIZ is None:
    raise SystemExit("No encuentro el proyecto (busque amparo/nucleo/schema.py en: "
                     + ", ".join(map(str, CANDIDATAS)) + "). Revisa que Drive este montado "
                     "y que la carpeta agenteAmparo exista en la cuenta correcta.")
sys.path.insert(0, str(RAIZ))

In [ ]:
# Corrida, sello de Drive (`data/manifiesto.py --contra-sello`), clave de
# Groq (se pide por teclado si falta) y `arranque.celda("m3")`, en ese orden.
from amparo.pipelines import m3_rag as m3
ctx = m3.preparar(hay_gpu=HAY_GPU)

## 1 · Criterio 1 — Corpus: procedencia y responsabilidad

**De dónde salió.** Transcrito del normograma oficial de la Cancillería y de la
edición de la Rama Judicial de la Constitución, extraído **con script** y sin
que nadie lo retipeara: `data/reparar_corpus.py` y `data/auditar_corpus.py`
quedan en el repo y son re-ejecutables.

**Licencia.** Normas colombianas: documentos oficiales de dominio público,
Ley 23 de 1982 artículo 41.

**Quién responde.** El equipo. La transcripción es nuestra, no una copia
certificada.

**Qué pasa si el corpus está mal.** Un RAG sirve lo que recupera con apariencia
de autoridad. Por eso cada fragmento lleva URL, `vigencia`, `transcripcion` y
`verificado`, y los incisos inexequibles se **marcan en vez de borrarse**: el
modelo tiene que poder recuperar el art. 11 del Decreto 2591 y contradecir a
quien lo invoque como vigente.

In [ ]:
m3.corpus(ctx)

### Tabla de riesgos del corpus

| riesgo | ejemplo concreto aquí | qué le pasa al usuario si nadie lo detecta | cómo se detecta | cómo se mitiga |
|---|---|---|---|---|
| norma derogada citada como vigente | `D2591-11`, art. 11, **inexequible** por C-543/1992 | cree que su tutela caducó a los dos meses y no la presenta | `vigencia` distinto de "vigente" | se marca y se deja en el corpus para poder contradecirla; está **fuera** del whitelist de `normas.json` |
| paráfrasis servida como ley | en la v1.0.0 había cinco: `L1098-ALIM`, `L1437-NYR`, `L1581-SENS`… | cita como texto legal algo que escribió el equipo | `transcripcion` distinto de "literal" | las cinco se reemplazaron por transcripción literal; hoy el corpus es 100% literal |
| plazo mal entendido | Ley 1266 art. 13: **coexisten** 4 años desde el pago y 8 desde la mora | pide que le borren el reporte con el plazo equivocado | cotejo contra el texto oficial | `nota_vigencia` explica los dos disparadores |
| modificación no incorporada | art. 26 Ley 361: la copia oficial sigue diciendo "oficina de Trabajo" | invoca una autoridad que ya no es la competente | comparación con la norma modificatoria | marcado como **pendiente de verificar** en vez de afirmarlo |
| restos del conversor en el texto | marcadores "Jurisprudencia Unificación" y frases duplicadas en `L472-04` | lee como ley una etiqueta de navegación | `data/auditar_corpus.py` | auditoría re-ejecutable de los 57 fragmentos |
| cita sin respaldo en el corpus | `Ley 2157 de 2021`, `art. 42 CP`, `Ley 100 de 1993` | el sistema cita una norma que el RAG no puede mostrar | chequeo de cobertura en el auditor | las tres entraron; el chequeo corre en cada auditoría |
| sesgo de cobertura | 19 de 57 fragmentos son de la Constitución y 17 del Decreto 2591 | el sistema razona mejor sobre procedimiento que sobre derechos sustantivos | distribución por norma | declarado; es la debilidad principal del corpus |

**Fuentes descartadas y por qué.** Las copias en blogs y agregadores jurídicos
quedaron fuera porque no permiten verificar la vigencia; `secretariasenado.gov.co`
quedó fuera porque no respondió al momento de la cosecha, y se usó el
normograma de la Cancillería, que ya era la fuente de las leyes 1751, 1755 y 393.

In [ ]:
m3.auditar_corpus(ctx)

## 2 · Chunking: por artículo o por tamaño fijo

La hipótesis a batir es que el artículo es la unidad correcta, porque es la
unidad de cita en derecho colombiano: un fragmento que corta un artículo por la
mitad produce una cita que no se puede verificar.

Eso es una convicción hasta que se mide. La regla de decisión está fijada en
`rag.REGLA_CHUNKING` (`amparo/rag/recuperacion.py`), **antes** de correr.

In [ ]:
# Imprime la regla, compara los tres troceados y construye el indice con el elegido.
m3.chunking(ctx)

## 3 · Criterio 2 — Cinco técnicas de retrieval, con el delta medido

Lo único que cambia entre filas es el retrieval: mismo corpus, mismo chunking,
mismo k, mismas consultas de entrada.

- **Ingenuo**: embeddings densos, coseno, top-k. El baseline a batir.
- **Híbrido**: BM25 + denso fusionados con RRF. Fusiona por *puesto*, no por
  puntaje, así que no hay que normalizar escalas incomparables.
- **Reranking**: un cross-encoder reordena los candidatos leyendo consulta y
  fragmento juntos.
- **Transformación de consulta**: reescribe el relato del ciudadano en lenguaje
  jurídico antes de buscar. Es la que más debería rendir aquí, porque el vacío
  de vocabulario es el cuello de botella.
- **Multi-consulta**: tres reescrituras desde ángulos distintos, fusionadas con
  RRF. Basta con que una acierte el vocabulario de la norma.
- **HyDE**: genera el artículo hipotético que resolvería el caso y busca con
  ese texto.

La regla de decisión también está fijada de antemano, con una **guarda de
seguridad**: una técnica que recupere menos que el ingenuo queda descalificada
por elegante que sea.

In [ ]:
m3.recuperacion(ctx)

### Consultas fallidas

Los casos donde el retrieval no trajo ni un artículo esperado. Es la tarea de
S07 y el insumo de S08: cada fallo se le asigna una técnica.

In [ ]:
m3.consultas_fallidas(ctx)

## 4 · Criterio 3 — Tool use

Dos herramientas, y el **modelo decide cuál llamar**: no las dispara un `if`.

- `calcular_termino` — la herramienta externa. Un LLM no sabe que el 6 de enero
  se corre al lunes por la Ley 51 de 1983 ni que el Corpus Christi depende de la
  Pascua. Un plazo mal contado cambia la respuesta jurídica.
- `buscar_normas` — el retrieval como herramienta.

**Criterio de invocación**: va en el prompt del sistema, no solo aquí. Si no
está en el prompt no es un criterio, es un deseo.

**Nunca adivina**: si no puede calcular, devuelve `calculable: False` con el
motivo, y el sistema lo dice.

In [ ]:
m3.demo_terminos(ctx)

### Las cinco fallas del agente, provocadas a propósito

Documentar "esto nos pasó en una corrida" deja la cobertura al azar: si el
modelo no se equivoca ese día, la rama de manejo de la falla nunca se ejecuta y
nadie sabe si funciona. La autoprueba usa un cliente guionado, así que las cinco
se provocan de forma determinista y sin gastar un token.

In [ ]:
m3.fallas_agente(ctx)

## 5 · El motor de reglas

El artículo 6 del Decreto 2591 no es un criterio difuso: son **cinco causales
tasadas que se aplican en orden**. El artículo 9 de la Ley 393 es una regla de
prioridad. El artículo 4 de la Ley 472 es una lista cerrada.

Pedirle a un modelo que *adivine* el resultado de aplicar reglas escritas es lo
que produce el colapso a "tutela". Aquí el LLM extrae **hechos** y el motor
decide: el enrutamiento es determinista, auditable, y la alucinación en el campo
`fundamento` cae a cero por construcción.

In [ ]:
m3.motor_reglas(ctx)

## 6 · Los cuatro brazos

In [ ]:
# El modelo ajustado de M1, solo con GPU (brazos 2 y 3).
m3.modelos(ctx)

In [ ]:
# Los cuatro brazos, con checkpoint: volver a correr retoma donde quedo.
m3.correr_brazos(ctx)

## 7 · Criterio 4 — RAGAS y el harness, cruzados

Las cuatro métricas van implementadas en `amparo/rag/ragas_eval.py`. La razón no es
evitar una dependencia: es que **dos de las cuatro se pueden calcular de forma
exacta**, porque el campo `fundamento` del gold dice qué artículos debían
recuperarse. Una medida exacta es mejor que una estimada, y se reportan las dos
para contrastarlas: si divergen mucho, el sospechoso es el juez.

La versión exacta tiene además una ventaja práctica: **no gasta tokens**, así
que cubre el 100% de los casos y no una ventana que se corta cuando se acaba el
cupo de la API.

In [ ]:
m3.ragas_propio(ctx)

### La lectura cruzada: ¿los aciertos son del retrieval?

La pregunta incómoda: un caso puede acertar **sin que el contexto recuperado
respalde la respuesta**. Si eso pasa, parte del acierto no se puede atribuir al
retrieval y el sistema parece mejor de lo que es.

Se mide caso por caso con `context_recall` exacto, que no cuesta tokens.

In [ ]:
m3.cruce(ctx)

### Contraste con la librería `ragas` real

Las métricas implementadas a mano se contrastan con la librería sobre una
muestra. Si divergen, el sospechoso somos nosotros.

In [ ]:
m3.ragas_libreria(ctx)

## 8 · El harness de M2, sin un solo cambio

In [ ]:
m3.scorecards(ctx)

### 8.1 · Los cuatro brazos sobre LOS MISMOS casos

La tabla anterior pone en columnas contiguas cosas que no son comparables: el
brazo 1 se puntúa **solo** sobre el eval compartido —M1 nunca vio el eval de
M3— y los brazos 2 a 4 sobre el compartido **más** el de M3. Compararlos así
castiga a los brazos con RAG por haber sido evaluados además en un conjunto
más duro, y encima cada columna trae una clase mayoritaria distinta, así que
ni el listón es el mismo.

Esta celda vuelve a puntuar los cuatro **sobre los casos comunes**. Es la
tabla que hay que citar para decir cuál es mejor; la anterior sirve para ver
cómo se comporta cada brazo en todo lo que puede responder.

Los dos tamaños no se escriben aquí a propósito: cambian con cada versión del
eval set, y una cifra clavada en el markdown caduca sin que nadie se entere
—ya pasó en esta misma celda—. La salida de la celda los imprime, y la celda
de publicación los repite al lado de cada CSV diciendo cuál es cuál.

In [ ]:
m3.comunes(ctx)

In [ ]:
m3.comparar(ctx)

### 8.2 · Qué cuesta cada brazo

La tabla de calidad compara aciertos. Esta compara **lo que cuesta
conseguirlos**, y sin ella no se puede decidir nada: el brazo 4 tarda ~8 s por
caso contra ~0 del brazo 1, y eso no es un detalle de implementación — es la
conclusión.

Tres cosas que la media de latencia escondía:

- la **p50** frente a la media: el agente hace entre 1 y 5 pasos según el caso,
  así que su distribución tiene cola y la media la arrastran los casos que
  agotan el presupuesto;
- las **llamadas reales** frente a las de caché: una segunda corrida sale
  entera de caché y parece gratis, cuando el coste real es el de la primera;
- los **tokens**, que es lo que se factura y lo que topa el cupo de la API.

El brazo 1 aparece como `n/d`, no como 0: son predicciones de M1, su coste es
el de esa corrida, y un 0 se leería como «es gratis».

In [ ]:
m3.tabla_costos(ctx)

In [ ]:
# Escribe los CSV, comprueba TODOS los artefactos, publica en results/ y cierra la corrida.
m3.publicar(ctx)

## 9 · Dónde falla el sistema

Rellenar con los números de la corrida. Lo que hay que mirar y no maquillar:

1. **El delta de recuperación, con la guarda de seguridad.** ¿Qué técnica movió
   qué? Si alguna quedó descalificada por recuperar menos que el ingenuo, es un
   resultado, no un fracaso: HyDE y multi-consulta cuestan una llamada a la API
   por caso y hay que saber si la pagan.

2. **La latencia.** Está en la tabla de deltas y en la última columna del
   scorecard. Una técnica que gana dos puntos de recall y multiplica por 180 el
   tiempo de retrieval no entra al sistema.

3. **`acc_mecanismo` contra la clase mayoritaria.** Es el listón real. El brazo
   4 debería despegarse de forma clara, porque el mecanismo no lo predice: lo
   calcula una regla.

4. **Los adversariales.** La medida de si el sistema razona o reconoce formas.
   El salto del brazo 1 al 4 es el resultado central de M3.

5. **La abstención, en los dos sentidos.** `acierto_abstencion` sola no dice
   nada: un sistema que se calla siempre saca 100%. Hay que leerla junto a
   `abstencion_indebida`.

6. **Las dos métricas de cita.** `tasa_alucinacion` dice si la norma existe;
   `precision_citas` dice si viene al caso. Un sistema puede tener 0% de
   alucinación y citas inservibles.

7. **Los aciertos con `context_recall = 0`.** Son aciertos que no vienen del
   retrieval. En el brazo 4 la explicación es el motor de reglas, y hay que
   decirlo: el mérito es del art. 6 del Decreto 2591, no de la búsqueda.

8. **La abstención del brazo 4, que ahora existe.** Antes daba 0 de 6 por
   construcción: el motor siempre encontraba una vía
   (`sistema.DEFECTO_PRO_HOMINE`). Ahora el extractor responde además si hay
   un caso que triar, y si no lo hay el brazo emite la abstención del formato.
   El número que hay que mirar **no** es el acierto sobre los 6 —seis casos no
   validan ningún criterio de abstención— sino `abstencion_indebida` sobre los
   103 **con** respuesta correcta: valía 0.0, y si el criterio nuevo se
   abstiene de más, sube. Y el contador `abstencion_por_no_triable` dice
   cuántas veces se usó, para que la cifra no se lea sin saber de dónde sale.

9. **El presupuesto del agente.** `pasos_promedio` está **censurado por
   arriba**: un caso que se corta en el techo no aporta los pasos que habría
   necesitado, aporta el techo, así que la media estima una cota *inferior*.
   Con más de la mitad de la muestra tocando el techo no estima nada útil, y
   `casos_en_el_techo` es la cifra que hay que leer primero. El presupuesto
   cuenta ahora pasos **productivos** —una búsqueda cuyos fragmentos ya
   estaban todos en las observaciones anteriores no cobra— y hay un techo de
   iteraciones aparte, porque «dar vueltas sin aportar» y «necesitar más
   pasos» son cosas distintas y antes se contaban igual.

### Limitaciones, sin adornos

- **Corpus:** 69 fragmentos · 20 de la Constitución · 17 del Decreto 2591.
  Cubre los casos de evaluación y no la casuística real de la tutela
  colombiana, y está sesgado hacia procedimiento. (Esa línea la comprueba
  `data/auditar_notebooks.py` contra `data/corpus.json`: la versión anterior
  decía 57 fragmentos y 19 de la Constitución, y llevaba dos ampliaciones del
  corpus sin actualizarse.)
- **Eval:** 109 casos en el compartido y 21 en el de M3. Con ese tamaño una
  diferencia de 5 puntos está dentro del ruido. Los deltas grandes son
  fiables; los pequeños, no, y el intervalo de confianza de cada scorecard es
  lo que lo dice —no el orden de la tabla—.
- **Las etiquetas las puso un equipo sin formación jurídica.** 142 de las 160
  del dataset las deriva el motor de reglas —todas menos las 18 de abstención,
  que por diseño no llevan hechos—, lo que las hace **auditables, no
  correctas**. Ya apareció un error así (A09).
- **El brazo 4 depende de que la extracción de hechos sea correcta.** Si el LLM
  se equivoca al decir si existe otro medio judicial, la regla aplica
  impecablemente sobre un hecho falso. La decisión es auditable, que no es lo
  mismo que correcta.
- **Esto no es apto para uso real.** Es un ejercicio académico. Un sistema que
  decide si alguien accede a la justicia necesita validación de abogados.

## 10 · Cierre de la entrega · los cuatro pasos de después

M3 es el último de los tres, así que aquí se cierra todo. Los cuatro pasos van
en una sola celda **a propósito**: por separado es fácil correr uno y olvidar
los otros, y el que se olvida siempre es el que habría avisado de algo.

Ninguno de los cuatro escribe en `results/`, así que esta celda no puede
estropear la corrida que acaba de terminar. Lo peor que puede pasar es que
diga que algo no cuadra — que es su trabajo.

| paso | qué hace | si falla |
|---|---|---|
| `verificar_entrega` | que los 33 artefactos estén, sean de la misma corrida de su módulo y compartan huella de código | **lo importante**: las cifras no son del mismo experimento y no se pueden comparar |
| `reporte` | genera `REPORTE.md` leyendo los artefactos, sin escribir ninguna cifra a mano | se niega a escribir si falta un artefacto, en vez de dejar huecos silenciosos |
| `diagnostico_cumplimiento` | lee `m3_agente_trazas.json` y dice **cuál** de las 6 causas rompe `accion_cumplimiento` | queda acotado a 6 en vez de resuelto |
| `calibracion_juez --preparar` | escribe `data/calibracion_1.csv` con 10 casos para que una persona los califique | falta M1 o M2, y lo dice nombrando el archivo |

In [ ]:
m3.cierre(ctx)